# Chapter 1 Preprocessing Runbook

This notebook is a thin orchestration and QC layer for the canonical Chapter 1 preprocessing pipeline in this repository.

- It reads standardized upstream ASIC artifacts from explicit artifact paths.
- It uses the shared JSON run config that is also used by the CLI.
- It calls the package pipeline once, then displays the cohort, instance, label, and model-ready outputs stage by stage.
- It writes final outputs into this repo's artifact directory.


## Configuration

Edit the shared run config file before executing the notebook end to end. The default repo config points at the standardized ASIC artifact root in `icu-data-platform`.


In [ ]:
from pathlib import Path

RUN_CONFIG_PATH = Path("config") / "ch1_run_config.json"


In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "chapter1_mortality_decomposition").exists():
    if (PROJECT_ROOT.parent / "src" / "chapter1_mortality_decomposition").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise RuntimeError("Run this notebook from the repo root or the notebooks directory.")

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import chapter1_mortality_decomposition.artifacts as artifacts_module
import chapter1_mortality_decomposition.pipeline as pipeline_module
import chapter1_mortality_decomposition.run_config as run_config_module

artifacts_module = importlib.reload(artifacts_module)
pipeline_module = importlib.reload(pipeline_module)
run_config_module = importlib.reload(run_config_module)

load_chapter1_inputs = artifacts_module.load_chapter1_inputs
build_chapter1_dataset = pipeline_module.build_chapter1_dataset
write_chapter1_dataset = pipeline_module.write_chapter1_dataset
load_chapter1_run_config = run_config_module.load_chapter1_run_config

pd.set_option("display.max_colwidth", 160)

run_config = load_chapter1_run_config(PROJECT_ROOT / RUN_CONFIG_PATH)
if not hasattr(run_config, "split_random_seed"):
    raise AttributeError(
        "Loaded Chapter1RunConfig does not expose split_random_seed. Restart the kernel and rerun the notebook from the top."
    )

config = run_config.to_chapter1_config()
input_dir = run_config.input_dir
output_dir = run_config.output_dir

display(Markdown(f"**Project root:** `{PROJECT_ROOT}`"))
display(Markdown(f"**Run config:** `{run_config.run_config_path}`"))
display(Markdown(f"**Using run_config module:** `{Path(run_config_module.__file__).resolve()}`"))
display(
    pd.DataFrame(
        [
            {"setting": "input_dir", "value": str(input_dir)},
            {"setting": "input_format", "value": run_config.input_format},
            {"setting": "output_dir", "value": str(output_dir)},
            {"setting": "output_format", "value": run_config.output_format},
            {"setting": "feature_set_config_path", "value": str(run_config.feature_set_config_path)},
            {"setting": "horizons_hours", "value": list(run_config.horizons_hours)},
            {"setting": "min_required_core_groups", "value": run_config.min_required_core_groups},
            {"setting": "split_random_seed", "value": run_config.split_random_seed},
            {
                "setting": "mech_vent_stay_qc_artifact",
                "value": str(input_dir / "qc" / f"mech_vent_ge_24h_stay_level.{run_config.input_format}"),
            },
            {
                "setting": "mech_vent_episode_qc_artifact",
                "value": str(input_dir / "qc" / f"mech_vent_ge_24h_episode_level.{run_config.input_format}"),
            },
        ]
    )
)


In [ ]:
if not input_dir.exists():
    raise FileNotFoundError(
        f"Input directory does not exist: {input_dir}. Update {run_config.run_config_path}."
    )

inputs = load_chapter1_inputs(input_dir=input_dir, input_format=run_config.input_format)

artifact_overview = pd.DataFrame(
    [
        {"table": "static_harmonized", "rows": inputs.static_harmonized.shape[0], "columns": inputs.static_harmonized.shape[1]},
        {"table": "dynamic_harmonized", "rows": inputs.dynamic_harmonized.shape[0], "columns": inputs.dynamic_harmonized.shape[1]},
        {"table": "block_index", "rows": inputs.block_index.shape[0], "columns": inputs.block_index.shape[1]},
        {"table": "blocked_dynamic_features", "rows": inputs.blocked_dynamic_features.shape[0], "columns": inputs.blocked_dynamic_features.shape[1]},
        {"table": "stay_block_counts", "rows": inputs.stay_block_counts.shape[0], "columns": inputs.stay_block_counts.shape[1]},
        {"table": "mech_vent_stay_level_qc", "rows": inputs.mech_vent_stay_level_qc.shape[0], "columns": inputs.mech_vent_stay_level_qc.shape[1]},
        {"table": "mech_vent_episode_level", "rows": inputs.mech_vent_episode_level.shape[0], "columns": inputs.mech_vent_episode_level.shape[1]},
    ]
)
display(Markdown("## Loaded Upstream Artifacts"))
display(artifact_overview)


In [ ]:
dataset = build_chapter1_dataset(inputs, config=config)


In [ ]:
display(Markdown("## Cohort QC"))
display(pd.DataFrame([
    {
        "retained_hospitals": dataset.cohort.retained_hospitals.shape[0],
        "retained_stays": dataset.cohort.table.shape[0],
    }
]))
display(Markdown("### Site-level eligibility"))
display(dataset.cohort.site_eligibility[["hospital_id", "site_included_ch1", "usable_core_vital_group_count", "exclusion_reason"]])
display(Markdown("### Stay-level exclusions"))
display(dataset.cohort.stay_exclusion_summary_by_hospital)
display(Markdown("### Canonical retained stay cohort"))
display(dataset.cohort.table)
display(Markdown("### Canonical cohort summary"))
display(dataset.cohort_summary)
display(Markdown("### Verification checks"))
display(dataset.verification_summary)


In [ ]:
display(Markdown("## Feature-set Configuration and Validation"))
display(dataset.feature_set_validation_summary[[
    "feature_set_name",
    "primary_feature_count",
    "extended_only_feature_count",
    "total_extended_feature_count",
    "available_in_blocked_schema_count",
    "missing_from_blocked_schema_count",
    "absent_from_retained_data_count",
    "missing_from_blocked_schema_features",
    "absent_from_retained_data_features",
]])


In [ ]:
display(Markdown("## Valid Prediction Instances"))
display(dataset.valid_instances.counts_by_horizon)
display(dataset.valid_instances.exclusion_summary)

display(Markdown("## Proxy Horizon Labels"))
display(dataset.labels.summary_by_horizon)
display(dataset.labels.unlabeled_reason_summary)


In [ ]:
display(Markdown("## Carry-forward and Missingness QC"))
for feature_set_name, feature_set_dataset in dataset.feature_sets.items():
    display(Markdown(f"### {feature_set_name.title()} Carry-forward Summary"))
    display(feature_set_dataset.model_ready.locf_feature_summary)
    display(feature_set_dataset.model_ready.ventilator_locf_summary)
    display(feature_set_dataset.model_ready.missingness_by_hospital_and_family)
    display(feature_set_dataset.model_ready.carry_forward_verification_summary)

display(Markdown("## Model-ready Datasets"))
final_qc_rows = []
for feature_set_name, feature_set_dataset in dataset.feature_sets.items():
    display(Markdown(f"### {feature_set_name.title()} Feature Set"))
    display(feature_set_dataset.model_ready.readiness_summary)
    display(feature_set_dataset.model_ready.split_summary)
    display(feature_set_dataset.model_ready.split_verification_summary)
    readiness = feature_set_dataset.model_ready.readiness_summary.set_index("metric")
    final_qc_rows.append(
        {
            "feature_set_name": feature_set_name,
            "model_ready_rows_total": int(feature_set_dataset.model_ready.table.shape[0]),
            "selected_feature_columns_total": readiness.at["selected_feature_columns_total", "value"],
            "configured_base_features_total": readiness.at["configured_base_features_total", "value"],
            "distinct_splits_total": readiness.at["distinct_splits_total", "value"],
            "global_median_imputation_applied_in_export": readiness.at["global_median_imputation_applied_in_export", "value"],
        }
    )
display(Markdown("### Primary vs Extended Summary"))
display(pd.DataFrame(final_qc_rows))


In [ ]:
output_paths = write_chapter1_dataset(
    dataset,
    output_dir=output_dir,
    output_format=run_config.output_format,
)

display(Markdown("## Outputs Written"))
output_manifest = pd.DataFrame(
    {
        "artifact_key": list(output_paths.keys()),
        "path": [str(path) for path in output_paths.values()],
    }
)
display(output_manifest)

print(f"Saved {len(output_paths)} output tables to {output_dir}")
